# Vector-DB Comparison — Flat, HNSW, IVF, BM25

Different retrieval tasks call for different index structures:

| Index | Exact? | Build time | Query time | Best for |
|---|---|---|---|---|
| **Flat** (brute-force cosine) | ✅ | O(N) | O(N·d) | < 10K docs or evaluation baseline |
| **HNSW** | ≈ | O(N log N) | O(log N) | Large corpora, low latency, updates |
| **IVF** | ≈ | O(N) train | O(nprobe·k·d) | Very large corpora, memory-constrained |
| **BM25** | ✅ | O(N) | O(|vocab|) | Text corpora, keyword-heavy queries |

This notebook benchmarks all four on the 10-doc synthetic corpus using the shared golden Q&A. No servers required — HNSW and IVF are built from scratch in numpy/sklearn.

In [ ]:
import os, sys, pathlib
ROOT = pathlib.Path.cwd()
while not (ROOT / 'shared').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
if not (os.getenv('OPENAI_API_KEY') or os.getenv('ANTHROPIC_API_KEY')):
    os.environ.setdefault('LLM_CACHE_ONLY', '1')
print('LLM_CACHE_ONLY =', os.environ.get('LLM_CACHE_ONLY', '0'))


In [ ]:
import time
import numpy as np
from shared.embedders import cosine_topk, hash_embed
from shared.loaders import load_corpus, load_golden_qa

DOCS = load_corpus()
QA   = [q for q in load_golden_qa() if q.source_ids]
doc_texts = [d.title + '. ' + d.abstract for d in DOCS]
doc_ids   = [d.arxiv_id for d in DOCS]
DIMS = 256
doc_vecs = hash_embed(doc_texts, dims=DIMS, seed=0)
print('docs:', len(DOCS), '  Qs:', len(QA), '  vecs:', doc_vecs.shape)

## Index 1 — Flat (brute-force)

Exact cosine via numpy dot product. O(N·d) per query.

In [ ]:
class FlatIndex:
    def __init__(self, vecs):
        self.vecs = vecs / (np.linalg.norm(vecs, axis=1, keepdims=True) + 1e-9)
    def search(self, qv, k=5):
        scores = self.vecs @ qv
        idx = np.argpartition(-scores, min(k, len(scores)-1))[:k]
        return idx[np.argsort(-scores[idx])]

t0 = time.perf_counter()
flat = FlatIndex(doc_vecs)
flat_build_ms = (time.perf_counter() - t0) * 1000
print(f'Flat build: {flat_build_ms:.2f} ms')

## Index 2 — Scratch HNSW

A simplified HNSW: 2 layers, greedy entry-point descent. Real HNSW uses `log(N)` layers; this version is pedagogical but shows the core idea.

In [ ]:
class ScratchHNSW:
    """Minimal HNSW with M=4 (max neighbours per node), 2 layers."""
    def __init__(self, vecs, m=4, max_layers=2, seed=42):
        rng = np.random.default_rng(seed)
        self.vecs = vecs / (np.linalg.norm(vecs, axis=1, keepdims=True) + 1e-9)
        n = len(vecs)
        # layers[lyr] = adjacency list  (dict node -> [neighbours])
        self.layers = [{} for _ in range(max_layers)]
        self.m = m
        self.ep = 0  # global entry point
        for i in range(n):
            level = int(rng.integers(0, max_layers))
            self._insert(i, level)

    def _sim(self, a, b):
        return float(self.vecs[a] @ self.vecs[b])

    def _insert(self, node, top_level):
        for lyr in range(top_level + 1):
            candidates = list(self.layers[lyr].keys())[:self.m]
            if candidates:
                candidates.sort(key=lambda x: -self._sim(node, x))
                neighbours = candidates[:self.m]
            else:
                neighbours = []
            self.layers[lyr][node] = neighbours
            for nb in neighbours:
                self.layers[lyr].setdefault(nb, [])
                if node not in self.layers[lyr][nb]:
                    self.layers[lyr][nb].append(node)

    def search(self, qv, k=5, ef=16):
        qv = qv / (np.linalg.norm(qv) + 1e-9)
        candidates = {self.ep}
        visited = set()
        best = []
        for lyr in range(len(self.layers) - 1, -1, -1):
            changed = True
            while changed:
                changed = False
                for c in list(candidates):
                    if c in visited: continue
                    visited.add(c)
                    score = float(self.vecs[c] @ qv)
                    best.append((score, c))
                    for nb in self.layers[lyr].get(c, []):
                        if nb not in visited:
                            candidates.add(nb)
                            changed = True
        best.sort(reverse=True)
        return np.array([c for _, c in best[:k]])

t0 = time.perf_counter()
hnsw = ScratchHNSW(doc_vecs, m=4, max_layers=2)
hnsw_build_ms = (time.perf_counter() - t0) * 1000
print(f'Scratch HNSW build: {hnsw_build_ms:.2f} ms')

## Index 3 — IVF (sklearn k-means centroids)

In [ ]:
from sklearn.cluster import MiniBatchKMeans

class IVFIndex:
    def __init__(self, vecs, n_clusters=3):
        self.vecs = vecs / (np.linalg.norm(vecs, axis=1, keepdims=True) + 1e-9)
        n_clusters = min(n_clusters, len(vecs))
        self.km = MiniBatchKMeans(n_clusters=n_clusters, random_state=0, n_init='auto')
        self.labels = self.km.fit_predict(self.vecs)
        self.buckets = {c: np.where(self.labels == c)[0] for c in range(n_clusters)}

    def search(self, qv, k=5, nprobe=2):
        qv = qv / (np.linalg.norm(qv) + 1e-9)
        centroid_scores = self.km.cluster_centers_ @ qv
        top_clusters = np.argsort(-centroid_scores)[:nprobe]
        cands = np.concatenate([self.buckets[c] for c in top_clusters if c in self.buckets])
        if len(cands) == 0:
            return np.array([], dtype=int)
        scores = self.vecs[cands] @ qv
        top = np.argsort(-scores)[:k]
        return cands[top]

t0 = time.perf_counter()
ivf = IVFIndex(doc_vecs, n_clusters=3)
ivf_build_ms = (time.perf_counter() - t0) * 1000
print(f'IVF build: {ivf_build_ms:.2f} ms')

## Index 4 — BM25

In [ ]:
import re
from rank_bm25 import BM25Okapi

TOKEN_RE = re.compile(r'[A-Za-z0-9]+')
def tokenize(t):
    return [w.lower() for w in TOKEN_RE.findall(t)]

t0 = time.perf_counter()
bm25_idx = BM25Okapi([tokenize(t) for t in doc_texts])
bm25_build_ms = (time.perf_counter() - t0) * 1000
print(f'BM25 build: {bm25_build_ms:.2f} ms')

def bm25_search(q, k=5):
    scores = bm25_idx.get_scores(tokenize(q))
    return np.argsort(-scores)[:k]

## Benchmark — recall@{1,3,5} across all four indexes

In [ ]:
def recall_at(index_fn, k, use_bm25=False):
    hits = 0
    for item in QA:
        qv = hash_embed([item.question], dims=DIMS, seed=0)[0]
        idx = bm25_search(item.question, k) if use_bm25 else index_fn(qv, k)
        got = {doc_ids[i] for i in idx if i < len(doc_ids)}
        if got & set(item.source_ids):
            hits += 1
    return round(hits / len(QA), 4)

results = {}
for name, fn in [('flat', flat.search), ('hnsw', hnsw.search), ('ivf', ivf.search)]:
    results[name] = {f'recall@{k}': recall_at(fn, k) for k in (1, 3, 5)}
results['bm25'] = {f'recall@{k}': recall_at(None, k, use_bm25=True) for k in (1, 3, 5)}

print(f'{'Index':<8}  {'recall@1':>9}  {'recall@3':>9}  {'recall@5':>9}  {'build_ms':>10}')
print('-' * 52)
for nm, v in results.items():
    bms = {'flat': flat_build_ms, 'hnsw': hnsw_build_ms, 'ivf': ivf_build_ms, 'bm25': bm25_build_ms}
    print(f'{nm:<8}  {v["recall@1"]:9.4f}  {v["recall@3"]:9.4f}  {v["recall@5"]:9.4f}  {bms[nm]:10.2f}')

## What you can extend

* Swap the scratch HNSW for `hnswlib` (Python binding to the original C++) to see how fast production HNSW is.
* For the IVF index, set `nprobe=n_clusters` (full scan) and verify recall matches flat.
* Add a column for index RAM footprint (`sys.getsizeof`).
* Point the BM25 index at a 500-doc corpus to see where the performance gap grows.